In [ ]:
# import pandas as pd

# train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
# sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
 
# print("Train shape:", train.shape)
# print("Test shape:", test.shape)
# print(train.head())
 
# most_common = train["answer"].value_counts().index.tolist()
# fallback_order = ["A", "B", "C", "D", "E"]
# for letter in fallback_order:
#     if letter not in most_common:
#         most_common.append(letter)
 
# top3 = " ".join(most_common[:3])
# print("Predicting top-3 for every row as:", top3)
 
# submission = pd.DataFrame({
#     "id": test["id"],
#     "answer" if "answer" in sample_sub.columns else sample_sub.columns[1]: top3
# })
 
# # Match sample_submission's exact column names/order
# submission.columns = sample_sub.columns
# submission.to_csv("submission.csv", index=False)
 
# print("\nSaved submission.csv")
# print(submission.head())

In [1]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
import pandas as pd
import numpy as np
import string
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Q1. Frequency distribution of correct answers

In [5]:
answer_counts = train['answer'].value_counts()
print(answer_counts)

most_freq = answer_counts.max()
least_freq = answer_counts.min()

result = most_freq + least_freq
print("Sum:", result)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Sum: 814


## Q2. Vocabulary size after lowercase + punctuation removal

In [6]:
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

train['clean_prompt'] = train['prompt'].apply(clean_text)

all_words = " ".join(train['clean_prompt']).split()
vocab = set(all_words)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 859


## Q3. Row ID 1 stopword filtering count

(Assuming id == 1, otherwise use .iloc[0])

In [7]:
row1 = train[train['id'] == 1]['clean_prompt'].values[0]

tokens = row1.split()
filtered_tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]

print("Words left after stopword removal:", len(filtered_tokens))
print(filtered_tokens)

Words left after stopword removal: 13
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


### If ID column doesn’t exist:

In [8]:
row1 = train.iloc[0]['clean_prompt']

## Q4. TF-IDF feature columns from prompt + options

In [9]:
combined_text = (
    train['prompt'].fillna('') + " " +
    train['A'].fillna('') + " " +
    train['B'].fillna('') + " " +
    train['C'].fillna('') + " " +
    train['D'].fillna('') + " " +
    train['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(combined_text)

print("Total feature columns:", X.shape[1])

Total feature columns: 2762


## Q5. Cosine similarity between prompt and option A for Row ID 1

In [10]:
prompt = train.iloc[0]['prompt']
option_a = train.iloc[0]['A']

prompt_vec = vectorizer.transform([prompt])
a_vec = vectorizer.transform([option_a])

sim = cosine_similarity(prompt_vec, a_vec)[0][0]

print("Similarity:", round(sim, 4))

Similarity: 0.272


## Q6. Highest similarity matches correct answer %

In [11]:
correct = 0
total = len(train)

for _, row in train.iterrows():
    prompt_vec = vectorizer.transform([row['prompt']])

    sims = {}
    for option in ['A', 'B', 'C', 'D', 'E']:
        option_vec = vectorizer.transform([row[option]])
        sims[option] = cosine_similarity(prompt_vec, option_vec)[0][0]

    predicted = max(sims, key=sims.get)

    if predicted == row['answer']:
        correct += 1

accuracy = (correct / total) * 100
print("Match Percentage:", round(accuracy, 2), "%")

Match Percentage: 13.55 %


## Q7. MAP@3 if truth = C, prediction = C A B

In [12]:
truth = 'C'
preds = ['C', 'A', 'B']

if truth in preds:
    score = 1 / (preds.index(truth) + 1)
else:
    score = 0

print(score)

1.0


## Q8. MAP@3 if truth = B, prediction = D B E

In [13]:
truth = 'B'
preds = ['D', 'B', 'E']

if truth in preds:
    score = 1 / (preds.index(truth) + 1)
else:
    score = 0

print(score)

0.5


## Q9. Majority Class Baseline MAP@3

In [14]:
top3 = train['answer'].value_counts().index[:3].tolist()
print("Top 3 Majority Classes:", top3)

def map3(actual, preds):
    if actual in preds:
        return 1 / (preds.index(actual) + 1)
    return 0

scores = train['answer'].apply(lambda x: map3(x, top3))

final_score = scores.mean()
print("Majority Baseline MAP@3:", final_score)

Top 3 Majority Classes: ['B', 'C', 'A']
Majority Baseline MAP@3: 0.42125


## Q10. TF-IDF Pipeline MAP@3

In [15]:
def map3(actual, preds):
    if actual in preds:
        return 1 / (preds.index(actual) + 1)
    return 0

scores = []

for _, row in train.iterrows():
    prompt_vec = vectorizer.transform([row['prompt']])

    sims = {}

    for option in ['A', 'B', 'C', 'D', 'E']:
        option_vec = vectorizer.transform([row[option]])
        sims[option] = cosine_similarity(prompt_vec, option_vec)[0][0]

    ranked = sorted(sims.items(), key=lambda x: x[1], reverse=True)
    top3_preds = [x[0] for x in ranked[:3]]

    scores.append(map3(row['answer'], top3_preds))

final_map3 = np.mean(scores)
print("TF-IDF Pipeline MAP@3:", final_map3)

TF-IDF Pipeline MAP@3: 0.2961666666666667
